<a href="https://colab.research.google.com/github/toecm/iedi-mas/blob/main/CA_IEDI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- INSTALL DEPENDENCIES ---
# We upgrade to 'google-genai' (the new SDK) and suppress warnings
!pip install -q openai-whisper rapidfuzz pandas gradio datasets transformers torchaudio torch librosa pydub ffmpeg-python jiwer google-genai python-dotenv requests yt-dlp soundfile

import warnings
# Suppress pydub syntax warnings specifically
warnings.filterwarnings("ignore", category=SyntaxWarning, module="pydub")

import os
import glob
import torch
import whisper
import pandas as pd
import requests
import tempfile
import yt_dlp
import random
import soundfile as sf
import shutil
import csv
from pydub import AudioSegment
from pydub.generators import Sine
from rapidfuzz import process, fuzz

# --- NEW GOOGLE SDK IMPORT ---
from google import genai
from google.genai import types

from datasets import load_dataset, Audio
import gradio as gr
from dotenv import load_dotenv
from threading import Lock
from huggingface_hub import HfApi, hf_hub_download, upload_file
import json
import re
import traceback
import time
from datetime import datetime
import threading
import concurrent.futures

# --- CONFIGURATION ---
HF_REPO_ID = "toecm/IEDID"

load_dotenv()

# Try Loading Keys from Colab Secrets
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') or os.getenv("HF_TOKEN")
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY') or os.getenv("GOOGLE_API_KEY")
    os.environ["PINATA_JWT"] = userdata.get('PINATA_JWT') or os.getenv("PINATA_JWT")
except (ImportError, Exception):
    pass

HF_TOKEN = os.getenv("HF_TOKEN")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
PINATA_JWT = os.getenv("PINATA_JWT")

# --- DIRECTORY SETUP ---
DATASET_DIR = "/content/iuuy_datasets"
os.makedirs(DATASET_DIR, exist_ok=True)

# New: Dedicated folder for JSON profiles
PROFILES_DIR = "/content/lab_profiles"
os.makedirs(PROFILES_DIR, exist_ok=True)

# --- HELPER: Generate Warning Sound ---
def create_warning_beep():
    try:
        beep = Sine(1000).to_audio_segment(duration=500).apply_gain(5)
        path = os.path.join(tempfile.gettempdir(), "warning_beep.wav")
        beep.export(path, format="wav")
        return path
    except Exception as e:
        print(f"⚠️ Could not generate beep: {e}")
        return None

WARNING_BEEP_PATH = create_warning_beep()

# --- DYNAMIC MODEL MANAGER (UPDATED FOR YOUR AVAILABLE MODELS) ---
class GeminiManager:
    def __init__(self, api_key):
        self.api_key = api_key
        self.client = None
        if self.api_key:
            self.client = genai.Client(api_key=self.api_key)

        # MAPPED TO YOUR DIAGNOSTIC RESULTS
        self.model_pro = "gemini-2.5-pro"
        self.model_flash = "gemini-2.5-flash"
        self.last_used_model = "Idle"

        # Self-Diagnostic on Startup
        try:
            print(f"🧠 Gemini Manager Connecting to: {self.model_flash}...")
            # Run a tiny test to verify connection immediately
            self.client.models.generate_content(model=self.model_flash, contents="Test")
            print("✅ Connection Verified: Model found and active.")
        except Exception as e:
            print(f"❌ Connection Warning: {e}")

    def generate_fast(self, prompt):
        if not self.client: raise Exception("Google API Key not found.")
        self.last_used_model = f"{self.model_flash} (Fast)"
        return self.client.models.generate_content(model=self.model_flash, contents=prompt)

    def generate_smart(self, prompt):
        if not self.client: raise Exception("Google API Key not found.")
        try:
            self.last_used_model = f"{self.model_pro} (Boost)"
            return self.client.models.generate_content(model=self.model_pro, contents=prompt)
        except Exception as e:
            print(f"⚠️ Pro-Boost Failed ({str(e)[:50]}...). Falling back to Flash.")
            self.last_used_model = f"{self.model_flash} (Fallback)"
            time.sleep(1)
            return self.client.models.generate_content(model=self.model_flash, contents=prompt)

    def get_status_string(self):
        icon = "🚀" if "pro" in self.last_used_model else "⚡"
        return f"{icon} Last Action: {self.last_used_model}"

gemini_manager = GeminiManager(GOOGLE_API_KEY) if GOOGLE_API_KEY else None

# --- HUGGING FACE SYNC MANAGER ---
class HFManager:
    def __init__(self):
        self.api = HfApi(token=HF_TOKEN)
        self.lock = Lock()

    def pull_datasets(self):
        print("⬇️ Pulling datasets & profiles from Hugging Face...")
        try:
            files = self.api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset")

            # 1. Pull CSVs to Dataset Dir
            csv_files = [f for f in files if f.endswith(".csv")]
            if not csv_files:
                seed_initial_data()
            else:
                for file in csv_files:
                    hf_hub_download(repo_id=HF_REPO_ID, filename=file, repo_type="dataset", local_dir=DATASET_DIR, token=HF_TOKEN)

            # 2. Pull JSONs to Profiles Dir
            json_files = [f for f in files if f.endswith(".json")]
            for file in json_files:
                # We save JSONs to the dedicated PROFILES_DIR
                hf_hub_download(repo_id=HF_REPO_ID, filename=file, repo_type="dataset", local_dir=PROFILES_DIR, token=HF_TOKEN)

        except Exception as e:
            print(f"❌ HF Pull Error: {e}")
            seed_initial_data()

    def push_update(self, filepath, commit_msg="Update from IEDI-MAS"):
        # We wrap the actual upload in a thread so the UI doesn't freeze
        def _upload_task():
            filename = os.path.basename(filepath)
            print(f"⬆️ Background Sync Starting: {filename}...")
            try:
                self.api.upload_file(
                    path_or_fileobj=filepath,
                    path_in_repo=filename,
                    repo_id=HF_REPO_ID,
                    repo_type="dataset",
                    commit_message=commit_msg
                )
                print(f"✅ Background Sync Complete: {filename}")
            except Exception as e:
                print(f"❌ HF Push Error: {e}")

        # Fire and forget
        threading.Thread(target=_upload_task, daemon=True).start()

    def upload_audio_sample(self, audio_path, dialect):
        clean_dialect = dialect.strip()
        filename = os.path.basename(audio_path)
        hf_path = f"audio/{clean_dialect}/{filename}"
        try:
            self.api.upload_file(path_or_fileobj=audio_path, path_in_repo=hf_path, repo_id=HF_REPO_ID, repo_type="dataset", commit_message=f"Add audio sample for {clean_dialect}")
            return hf_path
        except Exception as e:
            print(f"❌ Audio Upload Error: {e}")
            return None

hf_manager = HFManager()

def seed_initial_data():
    initial_data = {
        "Nigerian English": [{
            "Utterance": "How far?",
            "Clarification": "How are you doing?",
            "Tone_Category": "Casual/Greeting",
            "Linguistic_Context": "Common pidgin greeting functioning like 'What's up?'",
            "Syntax_Pattern": r"\bhow\s?far\b",
            "Pragmatic_Analysis": "A phatic communion greeting that expects a reciprocal inquiry rather than a literal distance measurement.",
            "file_name": ""
        }]
    }
    for dialect, rows in initial_data.items():
        filepath = os.path.join(DATASET_DIR, f"{dialect}.csv")
        if not os.path.exists(filepath):
            df = pd.DataFrame(rows)
            df["Dialect"] = dialect
            df.to_csv(filepath, index=False)
            hf_manager.push_update(filepath, "Initial Seed")

hf_manager.pull_datasets()

# --- AGENT 1: INPUT (Whisper) ---
class AgentInput:
    def __init__(self, model_size="small"):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"👂 Agent 1 (Input) Online: Loading Whisper ({model_size}) on {device}...")
        self.model = whisper.load_model(model_size, device=device)

    def transcribe(self, audio_path, language="en"):
        if not audio_path: return []
        result = self.model.transcribe(audio_path, language=language)
        return [{"speaker": "Speaker", "text": seg["text"].strip(), "start": seg["start"], "end": seg["end"]} for seg in result["segments"]]

# --- AGENT 2: INTERPRETATION (OPTIMIZED PARALLEL) ---
class AgentInterpretation:
    def __init__(self, gemini_manager_instance=None):
        self.df = pd.DataFrame()
        self.lookup_list = []
        self.gemini_manager = gemini_manager_instance

        # Initialize Profiles
        self.seed_initial_profiles()
        # Default load the Trainer
        self.lab_profile = self.load_profile_by_name("NSL Lab Trainer.json")

        print("🧠 Agent 2 (Interpretation) Online: Parallel Processing Enabled.")
        self.refresh_knowledge_base()

    # --- JSON PROFILE MANAGEMENT (Unchanged) ---
    def seed_initial_profiles(self):
        defaults = {
            "NSL Lab Trainer.json": {
                "lab_name": "NSL Lab Trainer", "role": "Instructor", "jargon": {"NLP": "Natural Language Processing"}, "pragmatic_rules": [{"trigger": "verbose", "interpretation": "Please be concise.", "tone": "Instruction"}]
            },
            "American Persona.json": {
                "lab_name": "American English",
                "cultural_context": "Low context, individualistic.",
                "jargon": {
                   "Rain check": "Decline now, accept later.", "Touch base": "Brief contact.", "Heads up": "Warning.",
                   "Cold turkey": "Stopping abruptly.", "Shoot an email": "Send quickly.", "Hard stop": "Must leave time.",
                   "Loop in": "Add to convo.", "Play it by ear": "Improvise.", "Cut to the chase": "Get to the point."
                },
                "pragmatic_rules": [
                    {"trigger": "How are you?", "speaker_role": "Any", "interpretation": "Phatic greeting (Hello only).", "tone": "Casual"},
                    {"trigger": "We should do lunch soon", "speaker_role": "Acquaintance", "interpretation": "Polite goodbye, no lunch intended.", "tone": "Polite"},
                    {"trigger": "I hear what you're saying", "speaker_role": "Colleague", "interpretation": "I understand but I disagree.", "tone": "Dismissive"},
                    {"trigger": "Interesting", "speaker_role": "Any", "interpretation": "Often means 'That is weird/wrong'.", "tone": "Ambiguous"},
                    {"trigger": "Let's take this offline", "speaker_role": "Meeting Participant", "interpretation": "Stop talking about this now.", "tone": "Directive"}
                ]
            },
            "Nigerian Persona.json": {
                "lab_name": "Nigerian Cultural Context", "cultural_context": "High context, communal.",
                "jargon": {"Wahala": "Trouble/Stress", "Abeg": "Please"},
                "pragmatic_rules": []
            }
        }
        for filename, content in defaults.items():
            path = os.path.join(PROFILES_DIR, filename)
            if not os.path.exists(path):
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(content, f, indent=2)

    def get_available_profiles(self):
        files = glob.glob(os.path.join(PROFILES_DIR, "*.json"))
        return [os.path.basename(f) for f in files]

    def load_profile_by_name(self, filename):
        path = os.path.join(PROFILES_DIR, filename)
        default_profile = {"lab_name": "General Context", "jargon": {}, "pragmatic_rules": []}
        if os.path.exists(path):
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    self.lab_profile = data
                    return data
            except Exception as e:
                print(f"Error loading {filename}: {e}")
                return default_profile
        return default_profile

    def save_specific_profile(self, filename, json_str):
        if not filename.endswith(".json"): filename += ".json"
        path = os.path.join(PROFILES_DIR, filename)
        try:
            new_profile = json.loads(json_str)
            with open(path, "w", encoding="utf-8") as f:
                json.dump(new_profile, f, indent=2)
            self.lab_profile = new_profile
            self.refresh_knowledge_base()
            if 'hf_manager' in globals():
                hf_manager.push_update(path, commit_msg=f"Update Profile: {filename}")
                return f"✅ Saved & Synced: {filename}"
            else:
                return f"⚠️ Saved Locally Only (HF Offline): {filename}"
        except json.JSONDecodeError: return "❌ Invalid JSON Format"
        except Exception as e: return f"❌ Save Error: {e}"

    def get_current_profile_text(self):
        return json.dumps(self.lab_profile, indent=2)

    def refresh_knowledge_base(self):
        all_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
        df_list = []
        for filename in all_files:
            try:
                dialect_name = os.path.basename(filename).replace(".csv", "")
                temp_df = pd.read_csv(filename, encoding='utf-8-sig', on_bad_lines='skip')
                temp_df["Dialect"] = dialect_name
                df_list.append(temp_df)
            except Exception as e: print(f"⚠️ Error loading {filename}: {e}")

        if df_list:
            self.df = pd.concat(df_list, ignore_index=True)
            self.lookup_list = self.df["Utterance"].tolist()
        else:
            self.lookup_list = []

    # --- AI HELPERS ---
    def generate_single_pragmatics(self, text, dialect, tone):
        if not self.gemini_manager: return "LLM Offline"
        prompt = f"Briefly analyze the pragmatic intent of: '{text}' (Dialect: {dialect}, Tone: {tone}). One sentence only."
        try: return self.gemini_manager.generate_smart(prompt).text.strip()
        except Exception as e: return f"Analysis Failed: {str(e)[:20]}"

    def get_few_shot_examples(self, text, n=3):
        if self.df.empty: return ""
        choices = self.df["Utterance"].dropna().tolist()
        matches = process.extract(text, choices, limit=n)
        examples_str = "\nHere are examples of how we analyze similar phrases in this lab:\n"
        found_any = False
        for match_tuple in matches:
            idx = match_tuple[2]
            row = self.df.iloc[idx]
            if row["Pragmatic_Analysis"] and row["Pragmatic_Analysis"] != "---":
                found_any = True
                examples_str += f"- Input: '{row['Utterance']}' -> Analysis: {row['Pragmatic_Analysis']} (Tone: {row['Tone_Category']})\n"
        return examples_str if found_any else ""

    def generate_unknown_analysis(self, text):
        if not self.gemini_manager: return []
        profile_context = json.dumps(self.lab_profile.get("jargon", {}), indent=2)
        few_shot_context = self.get_few_shot_examples(text)
        prompt = f"""
        Analyze this utterance: "{text}"
        Context / Dictionary for reference: {profile_context}
        {few_shot_context}
        Task:
        1. Identify if any jargon from the dictionary matches.
        2. Provide 3 interpretations following the style of the examples above.
        Output Strictly JSON:
        [ {{ "dialect": "...", "clarification": "...", "tone": "...", "context": "...", "pragmatics": "..." }} ]
        """
        try:
            response = self.gemini_manager.generate_smart(prompt)
            clean_text = re.sub(r"```json|```", "", response.text).strip()
            return json.loads(clean_text)
        except Exception as e:
            return [{"dialect": "Unknown", "clarification": "Hypothesis Failed", "tone": "---", "context": "---", "pragmatics": f"Error: {str(e)[:50]}..."}]

    def adapt_with_ai(self, full_text, db_row):
        # This is the slow function we will parallelize
        if not self.gemini_manager:
            return db_row["Clarification"], db_row["Pragmatic_Analysis"]

        prompt = f"""
        Reference Term: "{db_row['Utterance']}"
        Reference Meaning: "{db_row['Clarification']}"
        Reference Context: "{db_row['Linguistic_Context']}"
        Task: The user said: "{full_text}".
        This sentence contains the Reference Term, but the meaning might have shifted.
        Using the Reference Meaning as a guide, interpret the full sentence.
        Output valid JSON only: {{ "clarification": "...", "pragmatics": "..." }}
        """
        try:
            response = self.gemini_manager.generate_fast(prompt)
            clean_json = re.search(r"\{.*\}", response.text, re.DOTALL)
            if clean_json:
                data = json.loads(clean_json.group(0))
                return data.get("clarification", db_row["Clarification"]), data.get("pragmatics", "AI Adapted Analysis")
        except: pass
        return db_row["Clarification"], db_row["Pragmatic_Analysis"]

    # --- MAIN ANALYSIS FUNCTION (OPTIMIZED) ---
    def detect_and_analyze(self, text, threshold=80):
        results = []
        clean_text = text.lower().strip()
        seen_indices = set()

        # We use a ThreadPool to run AI tasks in parallel
        # "Max workers 5" means up to 5 AI calls can happen at the exact same time
        with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
            future_to_metadata = {} # Keeps track of which task belongs to which row

            # --- 1. SCANNING (Fast Local Search) ---
            if not self.df.empty:
                # Iterate ONCE through the DF to find candidates
                for index, row in self.df.iterrows():
                    match_found = False
                    match_type = "None"

                    # A. Check Regex
                    regex_str = str(row.get("Syntax_Pattern", "")).strip()
                    if len(regex_str) > 2 and regex_str not in ["---", "nan"]:
                        try:
                            match = re.search(regex_str, clean_text, re.IGNORECASE)
                            if match:
                                match_found = True
                                # If input is much longer than match, it's a "Partial" match -> Needs AI
                                if len(clean_text) > (len(match.group(0)) + 5): match_type = "Partial_Regex"
                                else: match_type = "Exact_Regex"
                        except: pass

                    # B. Check Phrase Scan (if Regex didn't hit)
                    if not match_found:
                        db_utterance = str(row["Utterance"]).strip().lower()
                        if len(db_utterance) > 2:
                            pattern = r"\b" + re.escape(db_utterance) + r"\b"
                            if re.search(pattern, clean_text):
                                match_found = True
                                if len(clean_text) > (len(db_utterance) + 5): match_type = "Partial_Scan"
                                else: match_type = "Exact_Scan"

                    # --- DECISION: Fast Result or Slow AI Task? ---
                    if match_found and index not in seen_indices:
                        seen_indices.add(index)

                        if "Partial" in match_type:
                            # SLOW PATH: Submit to background thread
                            # We pass the method 'self.adapt_with_ai', and arguments (text, row)
                            future = executor.submit(self.adapt_with_ai, text, row)
                            future_to_metadata[future] = {
                                "row": row,
                                "source": f"💎 DB Match + ✨ AI ({match_type})"
                            }
                        else:
                            # FAST PATH: Add immediately
                            results.append({
                                "Source": f"💎 Database ({match_type})",
                                "Dialect": row["Dialect"],
                                "Clarification": row["Clarification"],
                                "Tone": row.get("Tone_Category", "---"),
                                "Context": row.get("Linguistic_Context", "---"),
                                "Pragmatic Analysis": row.get("Pragmatic_Analysis", "---")
                            })

            # --- 2. FUZZY MATCH (Fast Local) ---
            # We run this locally while waiting for threads (it's fast enough)
            matches = process.extract(text, self.lookup_list, scorer=fuzz.ratio, limit=3)
            for best_utterance, score, index in matches:
                if score >= threshold and index < len(self.df) and index not in seen_indices:
                    seen_indices.add(index)
                    row = self.df.iloc[index]
                    results.append({
                        "Source": "🗄️ Database (Fuzzy)",
                        "Dialect": row["Dialect"],
                        "Clarification": row["Clarification"],
                        "Tone": row.get("Tone_Category", "---"),
                        "Context": row.get("Linguistic_Context", "---"),
                        "Pragmatic Analysis": row.get("Pragmatic_Analysis", "---")
                    })

            # --- 3. PROFILE CHECKS (Fast Local) ---
            if self.lab_profile:
                # Jargon
                for key, val in self.lab_profile.get("jargon", {}).items():
                    if re.search(r"\b" + re.escape(key.lower()) + r"\b", clean_text):
                        results.append({
                            "Source": "📒 Codebook (Jargon)", "Dialect": self.lab_profile.get("lab_name", "Custom"),
                            "Clarification": f"{key}: {val}", "Tone": "Contextual", "Context": "Lab Jargon", "Pragmatic Analysis": "Jargon Term Detected"
                        })
                # Rules
                for rule in self.lab_profile.get("pragmatic_rules", []):
                    trigger = rule.get("trigger", "").lower()
                    if trigger and trigger in clean_text:
                        results.append({
                            "Source": "📜 Codebook (Rule)", "Dialect": self.lab_profile.get("lab_name", "Custom"),
                            "Clarification": rule.get("interpretation", ""), "Tone": rule.get("tone", ""),
                            "Context": f"Role: {rule.get('speaker_role', 'Any')}", "Pragmatic Analysis": "Cultural Rule Triggered"
                        })

            # --- 4. COLLECT AI RESULTS (Wait for threads) ---
            for future in concurrent.futures.as_completed(future_to_metadata):
                meta = future_to_metadata[future]
                row = meta["row"]
                try:
                    clar, prag = future.result() # This blocks only until THIS task is done
                    results.append({
                        "Source": meta["source"],
                        "Dialect": row["Dialect"],
                        "Clarification": clar,
                        "Tone": row.get("Tone_Category", "---"),
                        "Context": row.get("Linguistic_Context", "---"),
                        "Pragmatic Analysis": prag
                    })
                except Exception as e:
                    print(f"AI Adaptation Task Failed: {e}")

        # --- 5. FALLBACK (If nothing found) ---
        if not results:
            ai_guesses = self.generate_unknown_analysis(text)
            for guess in ai_guesses:
                results.append({
                    "Source": "✨ AI Generated", "Dialect": guess.get("dialect", "Unknown"),
                    "Clarification": guess.get("clarification", "---"), "Tone": guess.get("tone", "---"),
                    "Context": guess.get("context", "---"), "Pragmatic Analysis": guess.get("pragmatics", "Hypothesis")
                })

        return results[:3]

    def get_rich_suggestions(self, text, dialect):
        if not self.gemini_manager or not text or not dialect: return []
        profile_context = json.dumps(self.lab_profile, indent=2)
        prompt = f"""interpret this {dialect} sentence: "{text}" using profile: {profile_context}. Output 3 JSON options: [{{ "clarification": "", "tone": "", "context": "", "pragmatics": "" }}]"""
        try:
            response = self.gemini_manager.generate_smart(prompt)
            clean_text = re.sub(r"```json|```", "", response.text).strip()
            return json.loads(clean_text)
        except: return []

    def generate_syntax_pattern(self, utterance):
        safe_pattern = r"\b" + re.escape(utterance.lower()) + r"\b"
        if not self.gemini_manager: return safe_pattern
        prompt = f"Create a Python Regex to capture variations of: '{utterance}'. Return ONLY the regex string."
        try:
            raw_pattern = self.gemini_manager.generate_fast(prompt).text.strip().replace("`", "")
            re.compile(raw_pattern)
            return raw_pattern
        except: return safe_pattern

# --- AGENT 4: TRUST (UPDATED FOR BATCHING) ---
class AgentTrust:
    def __init__(self):
        self.lock = Lock()
        print("🛡️ Agent 4 (Trust) Online: Asynchronous Saving Enabled.")

    def log_to_ipfs(self, data):
        if not PINATA_JWT: return "Local-Log-Only"
        headers = {"Authorization": f"Bearer {PINATA_JWT}"}
        try:
            # We don't return the result to the UI anymore in async mode, just print/log
            res = requests.post("https://api.pinata.cloud/pinning/pinJSONToIPFS", headers=headers, json=data)
            return res.json().get("IpfsHash", "Error")
        except: return "IPFS_Fail"

    def check_if_exists(self, utterance, dialect, brain_agent, clarification="", tone=""):
        if brain_agent.df.empty: return False
        clean_text = utterance.strip().lower()
        clean_dialect = dialect.strip()

        # Strict check to prevent duplicates
        match = brain_agent.df[
            (brain_agent.df["Utterance"].str.strip().str.lower() == clean_text) &
            (brain_agent.df["Dialect"].str.strip() == clean_dialect) &
            (brain_agent.df["Clarification"].str.strip() == clarification.strip())
        ]
        return not match.empty

    # --- METHOD 1: BATCH (For "Accept All" Button) ---
    def process_batch_feedback(self, dataframes, brain_agent, audio_path=None):
        """Iterates through all result tables and adds missing entries."""
        stats = {"added": 0, "verified": 0, "skipped": 0}

        # 1. Consolidate all rows from the 3 DataFrames
        all_rows = []
        for df in dataframes:
            if df is not None and not df.empty:
                for _, row in df.iterrows():
                    all_rows.append(row)

        # 2. Process each row
        def _batch_task():
            local_added = 0
            for row in all_rows:
                if row["Source"] == "---" or not row["Utterance"]: continue

                utterance = row["Utterance"]
                dialect = row["Dialect"]
                clar = row["Clarification"]
                tone = row["Tone"]
                context = row["Context"]
                prag = row["Pragmatic Analysis"]

            # 3. Check Existence
            exists = self.check_if_exists(utterance, dialect, brain_agent, clar, tone)

            if exists:
                # Just log the verification event to IPFS
                self.log_to_ipfs({
                    "action": "Verify_Existing",
                    "utterance": utterance,
                    "timestamp": pd.Timestamp.now().isoformat()
                })
                stats["verified"] += 1
            else:
                    # Note: Using safe default regex to speed up batching (skips LLM per row)
                    syntax = r"\b" + re.escape(utterance.lower()) + r"\b"
                    self.update_dataset_csv(dialect, utterance, clar, tone, context, syntax, audio_path, prag)
                    self.log_to_ipfs({"action": "Batch_Add", "utterance": utterance, "timestamp": pd.Timestamp.now().isoformat()})
                    local_added += 1

            if local_added > 0:
                print(f"🔄 Batch Background Task: Added {local_added} items. Refreshing Brain...")
                brain_agent.refresh_knowledge_base()

        # Run batch processing in background too
        threading.Thread(target=_batch_task, daemon=True).start()

        return f"✅ Batch Process Started in Background! The database will update shortly."

    # --- METHOD 2: SINGLE EDIT (Optimized for Speed) ---
    def process_feedback(self, action, original_text, dialect, clarification, tone, context, brain_agent, audio_path=None, pragmatics=""):
        # 1. Capture Data Immediately (Snapshot)
        timestamp = pd.Timestamp.now().isoformat()
        feedback_data = {
            "original": original_text, "dialect": dialect, "clarification": clarification,
            "tone": tone, "linguistic_context": context, "pragmatics": pragmatics, "action": action, "timestamp": timestamp
        }

        # 2. Define the Heavy Lifting as a Background Task
        def _background_save_task():
            try:
                print(f"⏳ Background Save Started for: {original_text[:20]}...")

                # A. Log to IPFS (Network)
                cid = self.log_to_ipfs(feedback_data)

                if action in ["Suggest Update", "Accept", "Force Overwrite"]:
                    # B. Generate Syntax (LLM Call - Slow)
                    # We check if brain_agent is available to avoid thread race conditions
                    syntax = r"\b" + re.escape(original_text.lower()) + r"\b"
                    if brain_agent:
                        try:
                            syntax = brain_agent.generate_syntax_pattern(original_text)
                        except: pass # Fallback to regex above if LLM fails

                    # C. Update CSV & Upload Audio (File I/O + Network)
                    self.update_dataset_csv(dialect, original_text, clarification, tone, context, syntax, audio_path, pragmatics)

                    # D. Refresh Memory (File I/O)
                    if brain_agent:
                        brain_agent.refresh_knowledge_base()

                    print(f"✅ Background Save Complete. IPFS: {cid}")
            except Exception as e:
                print(f"❌ Background Save Failed: {e}")
                traceback.print_exc()

        # 3. Fire and Forget
        threading.Thread(target=_background_save_task, daemon=True).start()

        # 4. Return immediately to UI
        return f"✅ Saved! (Processing in background...)\nAction: {action} recorded."

    # --- SHARED CSV WRITE LOGIC (Thread-Safe via Lock) ---
    def update_dataset_csv(self, dialect, utterance, clarification, tone, context, syntax, audio_path=None, pragmatics=""):
        clean_dialect = dialect.strip().title()
        if not clean_dialect.endswith("English") and not clean_dialect.endswith("Dialect"): clean_dialect += " Dialect"
        filepath = os.path.join(DATASET_DIR, f"{clean_dialect}.csv")

        # The Lock ensures multiple threads don't write to the CSV at the same time
        with self.lock:
            if not os.path.exists(filepath):
                new_df = pd.DataFrame(columns=["Utterance", "Dialect", "Clarification", "Tone_Category", "Linguistic_Context", "Syntax_Pattern", "Pragmatic_Analysis", "file_name"])
                new_df.to_csv(filepath, index=False)

            try:
                df = pd.read_csv(filepath, encoding='utf-8-sig', on_bad_lines='skip')
            except:
                df = pd.DataFrame(columns=["Utterance", "Dialect", "Clarification", "Tone_Category", "Linguistic_Context", "Syntax_Pattern", "Pragmatic_Analysis", "file_name"])

            for col in ["Tone_Category", "Linguistic_Context", "file_name", "Syntax_Pattern", "Pragmatic_Analysis", "Clarification"]:
                if col not in df.columns: df[col] = "---"

            final_audio = ""
            if audio_path and os.path.exists(audio_path):
                ext = os.path.splitext(audio_path)[1]
                unique_name = f"{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}_{random.randint(1000,9999)}{ext}"
                new_path = os.path.join(os.path.dirname(audio_path), unique_name)
                try:
                    shutil.copy2(audio_path, new_path)
                    # Note: This is now running in the background thread, so it won't freeze UI
                    final_audio = hf_manager.upload_audio_sample(new_path, dialect)
                except Exception as e:
                    print(f"Audio Save Error: {e}")
                    final_audio = "Error_Saving_Audio"

            new_row = pd.DataFrame([{
                "Utterance": utterance, "Dialect": clean_dialect, "Clarification": clarification,
                "Tone_Category": tone, "Linguistic_Context": context,
                "Pragmatic_Analysis": pragmatics,
                "Syntax_Pattern": syntax, "file_name": final_audio
            }])

            final_df = pd.concat([df, new_row], ignore_index=True)
            final_df.to_csv(filepath, index=False, quoting=csv.QUOTE_ALL)

            # Push to HF (Already threaded internally, but fine to call here)
            hf_manager.push_update(filepath, f"Update: {utterance}")
            return "Saved"

# --- AGENT 3: UX (UPDATED FOR BATCH BUTTON) ---
class AgentUX:
    def __init__(self, input_agent, brain_agent, trust_agent):
        self.input = input_agent
        self.brain = brain_agent
        self.trust = trust_agent
        self.last_audio_path = None
        self.suggestion_cache = {}
        print("🎨 Agent 3 (UX) Online: Building Interface...")

    def get_quota_status(self):
        if self.brain.gemini_manager: return self.brain.gemini_manager.get_status_string()
        return "Manager not active"

    def automated_pipeline(self, audio_path, language="en"):
        if not audio_path:
            empty = pd.DataFrame(columns=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"])
            return empty, empty, empty, "Waiting for Input...", self.get_quota_status()

        self.last_audio_path = audio_path
        segments = self.input.transcribe(audio_path, language)
        list_1, list_2, list_3 = [], [], []

        for seg in segments:
            raw = seg["text"]
            possible_interpretations = self.brain.detect_and_analyze(raw)
            def get_interp(idx):
                if idx < len(possible_interpretations): return possible_interpretations[idx]
                return {"Source": "---", "Dialect": "---", "Clarification": "---", "Tone": "---", "Context": "---", "Pragmatic Analysis": "---"}
            def make_row(interp):
                return {
                    "Source": interp["Source"], "Speaker": seg["speaker"], "Utterance": raw,
                    "Dialect": interp["Dialect"], "Clarification": interp["Clarification"],
                    "Tone": interp["Tone"], "Context": interp["Context"],
                    "Pragmatic Analysis": interp.get("Pragmatic Analysis", "---")
                }
            list_1.append(make_row(get_interp(0)))
            list_2.append(make_row(get_interp(1)))
            list_3.append(make_row(get_interp(2)))

        return pd.DataFrame(list_1), pd.DataFrame(list_2), pd.DataFrame(list_3), "✅ Analysis Complete", self.get_quota_status()

    def launch(self):
        existing_dialects = []
        if os.path.exists(DATASET_DIR):
            csv_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
            existing_dialects = [os.path.basename(f).replace(".csv", "") for f in csv_files]
        dropdown_choices = existing_dialects + ["+ Add New Dialect"]
        available_profiles = self.brain.get_available_profiles()

        custom_css = """
        #red_btn { background-color: #FF0000 !important; color: white !important; font-weight: bold; }
        """

        with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as ui:
            gr.Markdown("## 🌍 CA-IEDI: Active Listening & English Language Mediator")
            warning_player = gr.Audio(visible=False, autoplay=True)

            with gr.Tabs():
                with gr.Tab("🎙️ Live Analysis"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            audio_input = gr.Audio(label="Step 1: Speak/Upload", sources=["microphone", "upload"], type="filepath")
                            lang_select = gr.Dropdown(["en", "ko", "fr"], value="en", label="Step 2: Language (Optional)")
                            analyze_btn = gr.Button("Re-Run Analysis 🔄", variant="secondary")
                            quota_display = gr.Textbox(label="📊 Model Status", value=self.get_quota_status(), interactive=False)

                        with gr.Column(scale=3):
                            status_box = gr.Textbox(label="Status", interactive=False)
                            with gr.Row():
                                with gr.Column():
                                    gr.Markdown("### 🥇 Result 1")
                                    results_1 = gr.Dataframe(headers=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"], interactive=False, label="Option 1", type="pandas", wrap=True)
                                with gr.Column():
                                    gr.Markdown("### 🥈 Result 2")
                                    results_2 = gr.Dataframe(headers=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"], interactive=False, label="Option 2", type="pandas", wrap=True)
                                with gr.Column():
                                    gr.Markdown("### 🥉 Result 3")
                                    results_3 = gr.Dataframe(headers=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"], interactive=False, label="Option 3", type="pandas", wrap=True)

                    gr.Markdown("### ✍️ Active Feedback Loop")
                    # Note: We keep the manual inputs for "Suggest Update" but disconnect them from "Accept"
                    with gr.Row():
                        with gr.Column(scale=1):
                            orig_text_state = gr.Textbox(visible=True, label="Original Text (For Manual Edits)")
                            with gr.Row():
                                dialect_dropdown = gr.Dropdown(choices=dropdown_choices, label="Select Dialect", interactive=True)
                                new_dialect_input = gr.Textbox(label="Enter New Dialect Name", visible=False, interactive=True)
                        with gr.Column(scale=1):
                            suggestion_dropdown = gr.Dropdown(label="Suggest Clarification", choices=[], allow_custom_value=True, interactive=True)
                            selected_tone_state = gr.Textbox(label="Linguistic Tone", interactive=True)
                            selected_context_state = gr.TextArea(label="Linguistic Context", interactive=True, lines=2)
                            selected_pragmatics_state = gr.TextArea(label="Pragmatic Analysis", interactive=True, lines=2)

                    with gr.Row():
                        # CHANGED: "Accept" now says "Batch Accept All"
                        btn_accept = gr.Button("✅ Batch Accept / Verify All Results", variant="secondary")
                        btn_suggest = gr.Button("💾 Suggest Specific Edit", variant="primary")
                        btn_overwrite = gr.Button("⚠️ Confirm Overwrite", variant="stop", visible=False, elem_id="red_btn")

                    feedback_out = gr.Markdown()

                with gr.Tab("⚙️ Lab Context"):
                    gr.Markdown("### Profile Manager")
                    with gr.Row():
                        profile_selector = gr.Dropdown(choices=available_profiles, value="NSL Lab Trainer.json", label="Select Profile")
                        profile_filename = gr.Textbox(label="Filename (Edit to create new)", value="NSL Lab Trainer.json")
                    profile_editor = gr.Code(value=self.brain.get_current_profile_text(), language="json", label="Profile Content", lines=20)
                    save_profile_btn = gr.Button("💾 Save Profile", variant="primary")
                    profile_status = gr.Textbox(label="System Response", interactive=False)

            # --- EVENT LOGIC ---
            def update_suggestions_rich(text, dialect):
                # ... (Keep existing logic) ...
                try:
                    if not text or not dialect or dialect == "+ Add New Dialect":
                        return gr.update(choices=[]), "", "", "", self.get_quota_status()
                    suggestions_data = self.brain.get_rich_suggestions(text, dialect)
                    self.suggestion_cache = {}
                    display_choices = []
                    if not suggestions_data: return gr.update(choices=["No suggestions"]), "", "", "", self.get_quota_status()
                    for item in suggestions_data:
                        clar, tone, ctx = item.get("clarification", ""), item.get("tone", ""), item.get("context", "")
                        prag = item.get("pragmatics", "Auto-generated")
                        display_str = f"{clar}  [{tone}]"
                        display_choices.append(display_str)
                        self.suggestion_cache[display_str] = {"clar": clar, "tone": tone, "context": ctx, "pragmatics": prag}
                    if display_choices:
                        first = self.suggestion_cache[display_choices[0]]
                        return gr.update(choices=display_choices, value=display_choices[0]), first["tone"], first["context"], first["pragmatics"], self.get_quota_status()
                    return gr.update(choices=[]), "", "", "", self.get_quota_status()
                except: return gr.update(choices=["Error"]), "Error", "", "", self.get_quota_status()

            def on_suggestion_select(val):
                if val in self.suggestion_cache:
                    return self.suggestion_cache[val]["tone"], self.suggestion_cache[val]["context"], self.suggestion_cache[val]["pragmatics"]
                return "Custom", "User provided", ""

            dialect_dropdown.change(fn=update_suggestions_rich, inputs=[orig_text_state, dialect_dropdown], outputs=[suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state, quota_display])
            orig_text_state.blur(fn=update_suggestions_rich, inputs=[orig_text_state, dialect_dropdown], outputs=[suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state, quota_display])
            suggestion_dropdown.change(fn=on_suggestion_select, inputs=[suggestion_dropdown], outputs=[selected_tone_state, selected_context_state, selected_pragmatics_state])

            audio_input.change(self.automated_pipeline, [audio_input, lang_select], [results_1, results_2, results_3, status_box, quota_display])
            analyze_btn.click(self.automated_pipeline, [audio_input, lang_select], [results_1, results_2, results_3, status_box, quota_display])

            def handle_selection(evt: gr.SelectData, df):
                if df is None or len(df) == 0: return "", "", "", "", "", ""
                try:
                    row = df.iloc[evt.index[0]]
                    if row["Source"] == "---": return "", "", "", "", "", ""
                    d = row["Dialect"] if row["Dialect"] in existing_dialects else None
                    return row["Utterance"], d, row["Clarification"], row["Tone"], row["Context"], row["Pragmatic Analysis"]
                except: return "", "", "", "", "", ""

            results_1.select(handle_selection, [results_1], [orig_text_state, dialect_dropdown, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state])
            results_2.select(handle_selection, [results_2], [orig_text_state, dialect_dropdown, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state])
            results_3.select(handle_selection, [results_3], [orig_text_state, dialect_dropdown, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state])

            # --- SUBMISSION LOGIC ---

            # 1. NEW BATCH LOGIC
            def run_batch_accept(df1, df2, df3):
                return self.trust.process_batch_feedback([df1, df2, df3], self.brain, self.last_audio_path)

            # 2. Existing Manual Logic
            def check_and_submit_logic(orig, d_drop, d_new, clar_raw, tone, context, prag):
                try:
                    final_d = d_new.strip() if d_drop == "+ Add New Dialect" else d_drop
                    if not final_d or not orig: return "❌ Invalid Input", gr.update(visible=False), None
                    exists = self.trust.check_if_exists(orig, final_d, self.brain, clar_raw, tone)
                    if exists: return "⚠️ Entry already exists! Click 'Confirm Overwrite' to replace it.", gr.update(visible=True), WARNING_BEEP_PATH
                    else:
                        final_clar = str(clar_raw).rsplit("[", 1)[0].strip() if "[" in str(clar_raw) else clar_raw
                        audio_ref = self.last_audio_path
                        # We use process_feedback for manual single edits
                        msg = self.trust.process_feedback("Suggest Update", orig, final_d, final_clar, tone, context, self.brain, audio_ref, prag)
                        return msg, gr.update(visible=False), None
                except Exception as e: return f"❌ Error: {e}", gr.update(visible=False), None

            def force_overwrite_logic(orig, d_drop, d_new, clar_raw, tone, context, prag):
                try:
                    final_d = d_new.strip() if d_drop == "+ Add New Dialect" else d_drop
                    final_clar = str(clar_raw).rsplit("[", 1)[0].strip() if "[" in str(clar_raw) else clar_raw
                    audio_ref = self.last_audio_path
                    msg = self.trust.process_feedback("Force Overwrite", orig, final_d, final_clar, tone, context, self.brain, audio_ref, prag)
                    return msg, gr.update(visible=False), None
                except Exception as e: return f"❌ Error: {e}", gr.update(visible=True), None

            btn_suggest.click(check_and_submit_logic, [orig_text_state, dialect_dropdown, new_dialect_input, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state], [feedback_out, btn_overwrite, warning_player])
            btn_overwrite.click(force_overwrite_logic, [orig_text_state, dialect_dropdown, new_dialect_input, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state], [feedback_out, btn_overwrite, warning_player])

            # CHANGED: Accept button now connects to run_batch_accept
            btn_accept.click(run_batch_accept, [results_1, results_2, results_3], [feedback_out])

            def on_dialect_change(val): return gr.update(visible=True) if val == "+ Add New Dialect" else gr.update(visible=False)
            dialect_dropdown.change(on_dialect_change, inputs=dialect_dropdown, outputs=new_dialect_input)

            # Profile Management
            def change_profile(val):
                content = json.dumps(self.brain.load_profile_by_name(val), indent=2)
                return content, val
            def save_and_refresh_profile(filename, content):
                msg = self.brain.save_specific_profile(filename, content)
                new_list = self.brain.get_available_profiles()
                return msg, gr.update(choices=new_list, value=filename)
            profile_selector.change(change_profile, inputs=[profile_selector], outputs=[profile_editor, profile_filename])
            save_profile_btn.click(save_and_refresh_profile, inputs=[profile_filename, profile_editor], outputs=[profile_status, profile_selector])

        ui.launch(share=True, debug=True)

# --- START SYSTEM ---
agent1 = AgentInput()
agent2 = AgentInterpretation(gemini_manager)
agent4 = AgentTrust()
agent3 = AgentUX(agent1, agent2, agent4)
agent3.launch()

🧠 Gemini Manager Connecting to: gemini-2.5-flash...
✅ Connection Verified: Model found and active.
⬇️ Pulling datasets & profiles from Hugging Face...
👂 Agent 1 (Input) Online: Loading Whisper (small) on cpu...
🧠 Agent 2 (Interpretation) Online: Parallel Processing Enabled.
🛡️ Agent 4 (Trust) Online: Asynchronous Saving Enabled.
🎨 Agent 3 (UX) Online: Building Interface...


/tmp/ipython-input-3072668122.py:762: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as ui:
/tmp/ipython-input-3072668122.py:762: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as ui:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7d47ae4642a8672030.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7d47ae4642a8672030.gradio.live


In [ ]:
# DIAGNOSTIC: List my available models
from google import genai
client = genai.Client(api_key=GOOGLE_API_KEY)
print("🔍 Scanning available models...")
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(f" - {m.name}")

## 🚀 Getting Started with Hardhat

Hardhat is a development environment for compiling, deploying, testing, and debugging your Ethereum software. It helps developers manage and automate the recurring tasks that are inherent to building smart contracts and dApps.

### 1. Install Node.js and npm (if you don't have them)
Hardhat projects are typically set up using Node.js and its package manager, `npm`. You can download Node.js (which includes npm) from the official website: [nodejs.org](https://nodejs.org/en/download/).

### 2. Create a New Project Directory
It's best to create a dedicated directory for your Hardhat project.


In [ ]:
import os

project_name = "my-hardhat-project"
if not os.path.exists(project_name):
    os.makedirs(project_name)
    print(f"Created directory: {project_name}")
else:
    print(f"Directory '{project_name}' already exists.")

# Change to the new directory
%cd {project_name}

### 3. Initialize the Project and Install Hardhat

Inside your project directory, you'll initialize a new npm project and then install Hardhat locally.


In [ ]:
!npm init -y
!npm install --save-dev hardhat

### 4. Create a Hardhat Project

Now you can run the Hardhat command to create your first project. It will ask you to choose a project type (e.g., "Create a basic sample project"). You can select the default options.


In [ ]:
!npx hardhat

After running `npx hardhat`, you'll have a basic project structure with sample contracts, scripts, and tests. You can explore these files in the file browser (`/content/my-hardhat-project`).

### Next Steps:
*   **Explore `hardhat.config.js`**: This is where you configure your network, compilers, and plugins.
*   **Write Smart Contracts**: Look into the `contracts/` directory to start writing your Solidity code.
*   **Write Tests**: Use the `test/` directory to write tests for your contracts.
*   **Run Scripts**: The `scripts/` directory is for deployment and interaction scripts.

Let me know if you want to compile, deploy, or interact with a sample contract!

In [ ]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Load keys
load_dotenv()
PINATA_JWT = os.getenv("PINATA_JWT")

def fetch_ipfs_logs():
    if not PINATA_JWT:
        print("❌ Error: PINATA_JWT not found.")
        return

    print("🔍 Fetching pinned files from Pinata...")

    url = "https://api.pinata.cloud/data/pinList?status=pinned"
    headers = {"Authorization": f"Bearer {PINATA_JWT}"}

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        files = response.json().get('rows', [])

        print(f"✅ Found {len(files)} pinned logs.")

        all_logs = []

        for file in files:
            cid = file['ipfs_pin_hash']
            # Fetch content from a public gateway
            gateway_url = f"https://gateway.pinata.cloud/ipfs/{cid}"
            try:
                log_data = requests.get(gateway_url).json()
                # Add CID for reference
                log_data['ipfs_cid'] = cid
                all_logs.append(log_data)
                print(f"   -> Retrieved log: {cid}")
            except Exception as e:
                print(f"   ⚠️ Could not read content for {cid}: {e}")

        # Convert to DataFrame for easy viewing
        if all_logs:
            df = pd.DataFrame(all_logs)
            print("\n📊 Retrieved Data Summary:")
            print(df.head())

            # Save to CSV for analysis
            df.to_csv("ipfs_audit_trail.csv", index=False)
            print("\n💾 Saved full log to 'ipfs_audit_trail.csv'")
            return df
        else:
            print("No valid logs found.")

    except Exception as e:
        print(f"❌ API Error: {e}")

# Run the retrieval
audit_df = fetch_ipfs_logs()